## Data acquisition

This notebook downloads Mediterranean Sea Surface Temperature data from the Copernicus Marine Service, to be used for marine heatwave detection in the northwestern Mediterranean.

Dataset: Mediterranean Sea - High Resolution L4 Sea Surface Temperature Reprocessed (SST_MED_SST_L4_REP_OBSERVATIONS_010_021, DOI 10.48670/moi-00173). Daily, satellite-derived, optimally interpolated, 0.05 degree resolution, produced by CNR Italy.

## Study area and time window

The study area covers the northwestern Mediterranean: the Tuscan Archipelago, the Ligurian Sea and the Gulf of Lion (longitude 2 to 14 E, latitude 40.5 to 44.5 N). This region is documented by the Mare Caldo monitoring project (Greenpeace and DISTAV, University of Genoa) as one of the areas showing the strongest ecological impact from marine heatwaves in the Mediterranean, including recurring gorgonian mortality events. I am also a certified guide for the Tuscan Archipelago National Park, which sits at the centre of this bounding box.

Time window: 2016 to 2026, daily resolution. A shorter window than the full 1982-2026 record available, chosen to keep the dataset manageable while still covering enough years to build a climatological baseline for percentile thresholds.

## Test download: January 2024

Before downloading the full ten-year range, I ran a one-month test to check the bounding box, file size, and data format.

In [1]:
import xarray as xr

# Open the downloaded test file
ds = xr.open_dataset("../data/raw/test_jan2024.nc")

# Look at the structure: dimensions, variables, coordinate ranges
print(ds)

<xarray.Dataset> Size: 5MB
Dimensions:       (time: 31, latitude: 80, longitude: 239)
Coordinates:
  * time          (time) datetime64[ns] 248B 2024-01-01 ... 2024-01-31
  * latitude      (latitude) float32 320B 40.51 40.56 40.61 ... 44.42 44.47
  * longitude     (longitude) float32 956B 2.044 2.094 2.144 ... 13.9 13.95
Data variables:
    analysed_sst  (time, latitude, longitude) float64 5MB ...
Attributes: (12/51)
    Conventions:                CF-1.4 
    DSD_entry_id:               -GOS-L4HRfnd-MED
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    Scaling_Equation:           (scale_factor*data) + add_offset
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    ...                         ...
    time_coverage_end:          20240928T070000Z
    time_coverage_start:        20240927T190000Z
    title:                      Mediterranean Sea SST Analysis L4, Reprocesse...
    uuid:                     

In [2]:
# Check overall value range and NaN count across the full grid for day 1,
# to confirm land cells are masked as NaN and sea temperatures are plausible
sst_day1 = ds["analysed_sst"].isel(time=0)

print("Minimum value (K):", sst_day1.min().values)
print("Maximum value (K):", sst_day1.max().values)
print("Number of NaN:", sst_day1.isnull().sum().values)
print("Total number of cells:", sst_day1.size)

Minimum value (K): 283.2399936709553
Maximum value (K): 290.0199935194105
Number of NaN: 7184
Total number of cells: 19120



**Results**: 1.14 MB, grid of 80 x 239 cells (latitude x longitude), values ranging 283.24 to 290.02 K (10.1 to 16.9 C), consistent with winter sea temperatures in the western Mediterranean.

## Variable selection

The dataset includes four variables: analysed_sst (sea surface temperature), analysis_error (uncertainty of the estimate), mask (land/sea/ice flag), and sea_ice_fraction.

I checked the test file and confirmed that land cells (Corsica, coastlines) are already returned as NaN in analysed_sst, so a separate land/sea mask is not needed. sea_ice_fraction is not relevant for this region. analysis_error may be worth adding later if I want to weight results by measurement confidence, but it is not needed for the initial analysis. I am downloading analysed_sst only.

## Full download: 2016-2026

Same bounding box and variable as the test, extended to the full time window.

**Results**: 139.58 MB, in line with the size estimated from the one-month test. Saved to data/raw/med_sst_2016_2026.nc, excluded from version control via .gitignore.

## Verifying the full dataset

Before using this dataset for analysis, I repeat the same structural check on the complete 2016-2026 file, not just the test sample.

In [3]:
ds_full = xr.open_dataset("../data/raw/med_sst_2016_2026.nc")
print(ds_full)

n_nan = ds_full["analysed_sst"].isnull().sum().values
n_total = ds_full["analysed_sst"].size
print(f"Total NaN: {n_nan} ({n_nan/n_total*100:.2f}%)")

<xarray.Dataset> Size: 585MB
Dimensions:       (time: 3825, latitude: 80, longitude: 239)
Coordinates:
  * time          (time) datetime64[ns] 31kB 2016-01-01 ... 2026-06-21
  * latitude      (latitude) float32 320B 40.51 40.56 40.61 ... 44.42 44.47
  * longitude     (longitude) float32 956B 2.044 2.094 2.144 ... 13.9 13.95
Data variables:
    analysed_sst  (time, latitude, longitude) float64 585MB ...
Attributes: (12/51)
    Conventions:                CF-1.4 
    DSD_entry_id:               -GOS-L4HRfnd-MED
    Metadata_Conventions:       Unidata Dataset Discovery v1.0
    Scaling_Equation:           (scale_factor*data) + add_offset
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    ...                         ...
    time_coverage_end:          20240928T070000Z
    time_coverage_start:        20240927T190000Z
    title:                      Mediterranean Sea SST Analysis L4, Reprocesse...
    uuid:               

**Results**: 3825 daily records confirmed, same 80 x 239 grid as the test file. NaN cells are 37.57% of the total, consistent with the 37.6% observed in the January 2024 test, confirming the land/sea masking is uniform across the full time series.